# Count Segments in Multi-Session Subjects

This notebook counts segments from the annotation tables in `Infants_data/derivatives/NeuronicEEG/` and then reports the minimum segment count for subjects with 2, 3, and 4 sessions.

In [1]:
from pathlib import Path
import pandas as pd

base_dir = Path('Infants_data/derivatives/NeuronicEEG')


def count_segments(annotation_path: Path) -> int:
    """Return the number of segments in one annotations TSV file."""
    if not annotation_path.exists():
        return 0
    return len(pd.read_csv(annotation_path, sep='\t'))


subject_session_counts = {}
subjects_with_multiple_sessions = []

for subject_dir in sorted(base_dir.glob('sub-NORB*')):
    if not subject_dir.is_dir():
        continue

    sessions = sorted(
        [session_dir for session_dir in subject_dir.glob('ses-*') if session_dir.is_dir()],
        key=lambda path: int(path.name.split('-')[1]),
    )

    if len(sessions) <= 1:
        continue

    subjects_with_multiple_sessions.append(subject_dir.name)
    session_counts = {}

    for session_dir in sessions:
        session_name = session_dir.name
        annotation_path = (
            session_dir
            / 'eeg'
            / f'{subject_dir.name}_{session_name}_task-EEG_annotations.tsv'
        )
        session_counts[session_name] = count_segments(annotation_path)

    subject_session_counts[subject_dir.name] = session_counts

print('Segment counts for subjects with multiple sessions:')
for subject_id in sorted(subject_session_counts):
    counts_text = ', '.join(
        f'{session_name}={count}'
        for session_name, count in sorted(
            subject_session_counts[subject_id].items(),
            key=lambda item: int(item[0].split('-')[1]),
        )
    )
    print(f'{subject_id}: {counts_text}')

print('\nMinimum segments from session 1 and session 2:')
for session_total in (2, 3, 4):
    session_1_counts = []
    session_2_counts = []

    for subject_id, session_counts in subject_session_counts.items():
        if len(session_counts) != session_total:
            continue
        if 'ses-1' in session_counts:
            session_1_counts.append(session_counts['ses-1'])
        if 'ses-2' in session_counts:
            session_2_counts.append(session_counts['ses-2'])

    all_counts = session_1_counts + session_2_counts
    if all_counts:
        print(
            f'{session_total}-session subjects: '
            f'min(session 1)={min(session_1_counts) if session_1_counts else "n/a"}, '
            f'min(session 2)={min(session_2_counts) if session_2_counts else "n/a"}, '
            f'overall minimum={min(all_counts)}'
        )
    else:
        print(f'{session_total}-session subjects: no session 1/session 2 data found')

Segment counts for subjects with multiple sessions:
sub-NORB00064: ses-1=13, ses-2=20, ses-3=18
sub-NORB00065: ses-1=20, ses-2=20
sub-NORB00069: ses-1=15, ses-2=20, ses-3=20, ses-4=15
sub-NORB00070: ses-1=15, ses-2=20, ses-3=20
sub-NORB00074: ses-1=1, ses-2=20
sub-NORB00075: ses-1=20, ses-2=20, ses-3=20
sub-NORB00077: ses-1=20, ses-2=20
sub-NORB00078: ses-1=20, ses-2=20
sub-NORB00079: ses-1=15, ses-2=20
sub-NORB00080: ses-1=20, ses-2=20
sub-NORB00081: ses-1=15, ses-2=20
sub-NORB00082: ses-1=20, ses-2=1
sub-NORB00083: ses-1=20, ses-2=20
sub-NORB00084: ses-1=15, ses-2=20
sub-NORB00085: ses-1=8, ses-2=20
sub-NORB00086: ses-1=20, ses-2=20
sub-NORB00087: ses-1=20, ses-2=20
sub-NORB00089: ses-1=24, ses-2=20
sub-NORB00090: ses-1=15, ses-2=20
sub-NORB00094: ses-1=12, ses-2=20
sub-NORB00096: ses-1=20, ses-2=20
sub-NORB00103: ses-1=35, ses-2=20

Minimum segments from session 1 and session 2:
2-session subjects: min(session 1)=1, min(session 2)=1, overall minimum=1
3-session subjects: min(session